# Average Temperature

There are several values that can be cached. The first that can be cached is the size of each bin in the x and y directions:

In [3]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

def average_temperature(t):
    total_temp = 0
    for i in range(x_bins):
        for j in range(y_bins):
            x = (i + 0.5) * (x_bin_size)
            y = (j + 0.5) * (y_bin_size)
            total_temp = total_temp + 0.01 * x + 0.005 * y + 10 * math.sin(t / 365) + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

Average Temperature:  23.330580007602133


Timer unit: 1e-09 s

Total time: 1.63052 s
File: /tmp/ipykernel_16753/2024903876.py
Function: average_temperature at line 13

Line #      Hits         Time  Per Hit   % Time  Line Contents
    13                                           def average_temperature(t):
    14         1        904.0    904.0      0.0      total_temp = 0
    15      1001     253473.0    253.2      0.0      for i in range(x_bins):
    16   1001000  273778196.0    273.5     16.8          for j in range(y_bins):
    17   1000000  331626774.0    331.6     20.3              x = (i + 0.5) * (x_bin_size)
    18   1000000  342457100.0    342.5     21.0              y = (j + 0.5) * (y_bin_size)
    19   1000000  682166657.0    682.2     41.8              total_temp = total_temp + 0.01 * x + 0.005 * y + 10 * math.sin(t / 365) + 20
    20         1     235567.0 235567.0      0.0      print("Average Temperature: ", total_temp / (x_bins * y_bins))

Next, the value of `0.01 * x` is repeated many times in each loop:

In [4]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

def average_temperature(t):
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            y = (j + 0.5) * (y_bin_size)
            total_temp = total_temp + x_contribution + 0.005 * y + 10 * math.sin(t / 365) + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
Average Temperature:  23.330580007602133


Timer unit: 1e-09 s

Total time: 1.28465 s
File: /tmp/ipykernel_16753/4112077650.py
Function: average_temperature at line 13

Line #      Hits         Time  Per Hit   % Time  Line Contents
    13                                           def average_temperature(t):
    14         1       1148.0   1148.0      0.0      total_temp = 0
    15      1001     282040.0    281.8      0.0      for i in range(x_bins):
    16      1000     393310.0    393.3      0.0          x_contribution = (i + 0.5) * (x_bin_size) * 0.01
    17   1001000  269456224.0    269.2     21.0          for j in range(y_bins):
    18   1000000  343824948.0    343.8     26.8              y = (j + 0.5) * (y_bin_size)
    19   1000000  670514025.0    670.5     52.2              total_temp = total_temp + x_contribution + 0.005 * y + 10 * math.sin(t / 365) + 20
    20         1     182949.0 182949.0      0.0      print("Average Temperature: ", total_temp / (x_bins * y_bins))

Next, the time contribution stays the same in each loop, so we can cache that value too:

In [ ]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

def average_temperature(t):
    time_contribution = 10 * math.sin(t / 365)
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            y = (j + 0.5) * (y_bin_size)
            total_temp = total_temp + x_contribution + 0.005 * y + time_contribution + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
Average Temperature:  203.33058000536116


Timer unit: 1e-09 s

Total time: 0.977102 s
File: /tmp/ipykernel_16753/3336093631.py
Function: average_temperature at line 13

Line #      Hits         Time  Per Hit   % Time  Line Contents
    13                                           def average_temperature(t):
    14         1       4858.0   4858.0      0.0      time_contribution = math.sin(t / 365) + 20
    15         1        500.0    500.0      0.0      total_temp = 0
    16      1001     265263.0    265.0      0.0      for i in range(x_bins):
    17      1000     372017.0    372.0      0.0          x_contribution = (i + 0.5) * (x_bin_size) * 0.01
    18   1001000  251964768.0    251.7     25.8          for j in range(y_bins):
    19   1000000  327821398.0    327.8     33.6              y = (j + 0.5) * (y_bin_size)
    20   1000000  396431359.0    396.4     40.6              total_temp = total_temp + x_contribution + 0.005 * y + 10 * time_contribution
    21         1     241437.0 241437.0      0.0      print("Average Temperat

Next, the contribution from `y` is repeatedly calculated across different values of `x`. We can pre-compute all values and store them in a list so we simply need to access them:

In [7]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

y_contributions = []
for j in range(y_bins):
    y = (j + 0.5) * (y_bin_size)
    y_contributions.append(0.005 * y)

def average_temperature(t):
    time_contribution = 10 * math.sin(t / 365)
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            total_temp = total_temp + x_contribution + y_contributions[j] + time_contribution + 20
    print("Average Temperature: ", total_temp / (x_bins * y_bins))

%lprun -f average_temperature average_temperature(100)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
Average Temperature:  23.330580007602133


Timer unit: 1e-09 s

Total time: 0.770704 s
File: /tmp/ipykernel_16753/2285689593.py
Function: average_temperature at line 18

Line #      Hits         Time  Per Hit   % Time  Line Contents
    18                                           def average_temperature(t):
    19         1       5380.0   5380.0      0.0      time_contribution = 10 * math.sin(t / 365)
    20         1        435.0    435.0      0.0      total_temp = 0
    21      1001     287605.0    287.3      0.0      for i in range(x_bins):
    22      1000     509336.0    509.3      0.1          x_contribution = (i + 0.5) * (x_bin_size) * 0.01
    23   1001000  285861082.0    285.6     37.1          for j in range(y_bins):
    24   1000000  483858365.0    483.9     62.8              total_temp = total_temp + x_contribution + y_contributions[j] + time_contribution + 20
    25         1     182085.0 182085.0      0.0      print("Average Temperature: ", total_temp / (x_bins * y_bins))

The final optimisation in this sample solution is not actually a caching operation. Instead, it is a mathematical one. For each of the `x_bins * y_bins` bins we add the value `time_contribution + 20`, then divide the whole final answer by `x_bins * y_bins`. A more efficient approach would be to remove the `+ time_contribution + 20` from the loop and , instead, add it it to the final value instead:

In [10]:
import math
%load_ext line_profiler

x_extent = 100
y_extent = 50

x_bins = 1000
y_bins = 1000

x_bin_size = x_extent / x_bins
y_bin_size = y_extent / y_bins

y_contributions = []
for j in range(y_bins):
    y = (j + 0.5) * (y_bin_size)
    y_contributions.append(0.005 * y)

def average_temperature(t):
    time_contribution = 10 * math.sin(t / 365)
    total_temp = 0
    for i in range(x_bins):
        x_contribution = (i + 0.5) * (x_bin_size) * 0.01
        for j in range(y_bins):
            total_temp = total_temp + x_contribution + y_contributions[j] 
    print("Average Temperature: ", total_temp / (x_bins * y_bins) + time_contribution + 20)

%lprun -f average_temperature average_temperature(100)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
Average Temperature:  23.330580008037973


Timer unit: 1e-09 s

Total time: 0.698995 s
File: /tmp/ipykernel_16753/4224355143.py
Function: average_temperature at line 18

Line #      Hits         Time  Per Hit   % Time  Line Contents
    18                                           def average_temperature(t):
    19         1       5243.0   5243.0      0.0      time_contribution = 10 * math.sin(t / 365)
    20         1        462.0    462.0      0.0      total_temp = 0
    21      1001     294902.0    294.6      0.0      for i in range(x_bins):
    22      1000     553982.0    554.0      0.1          x_contribution = (i + 0.5) * (x_bin_size) * 0.01
    23   1001000  288013994.0    287.7     41.2          for j in range(y_bins):
    24   1000000  409937683.0    409.9     58.6              total_temp = total_temp + x_contribution + y_contributions[j] 
    25         1     189198.0 189198.0      0.0      print("Average Temperature: ", total_temp / (x_bins * y_bins) + time_contribution + 20)

# Morse Code

In this exercise, the `translate_word_to_morse` function will be called repeatedly with the same small number of words. This makes it a good place to add the `lru_cache` decorator to. The words that will be supplied as arguments are `North`, `South`, `East`, `West`, `0`, `10`, `20`, `30`, `40`, `50`, `60`, `70`, `80`, `90`, and `100`. There are fifteen of these words, so this is the minimum size of cache we should use.

In [ ]:
import cProfile
import random
from functools import lru_cache

@lru_cache(15)
def translate_word_to_morse(word):
    # This function translates a single word to morse code
    # Each letter is separated by a space
    morse_dict = {"a":".-", "b":"-...", "c":"-.-.", "d":"-..", "e":".", "f":"..-.", "g":"--.", "h":"....", "i":"..", "j":".---", "k":"-.-", "l":".-..", "m":"--", "n":"-.", "o":"---", "p":".--.", "q":"--.-", "r":".-.", "s":"...", "t":"-", "u":"..-", "v":"...-", "w":".--", "x":"-..-", "y":"-.--", "z":"--..", "1":".----", "2":"..---", "3":"...--", "4":"....-", "5":".....", "6":"-....", "7":"--...", "8":"---..", "9":"----.", "0":"-----"}
    morse_translation = ""
    for letter in word:
        morse_translation += morse_dict[letter] + " "
    return morse_translation.strip()

def translate_message_to_morse(message):
    # This function translates a full message to morse code
    # Each word is separated by three spaces
    morse_message = ""
    for word in message.split(" "):
        morse_message += translate_word_to_morse(word) + "   "
    return morse_message.strip()

def generate_random_message(length):
    # This first generates a message with length pairs of directions and distances
    # The direction will always be one of "north", "south", "east" and "west"
    # The distance will always be a multiple of 10 between 0 and 100
    directions = ["north", "east", "south", "west"]
    message = ""
    for i in range(length):
        direction = random.choice(directions)
        distance = 10 * random.randint(0, 10)
        message += direction + " " + str(distance) + " "
    return message

message = generate_random_message(10000)

cProfile.run('translate_message_to_morse(message)')

'sos' Translation:  ... --- ...
'sos help' Translation:  ... --- ...   .... . .-.. .--.
Length 5 message:  west 100 east 0 west 90 north 10 north 100 
         38 function calls in 0.160 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.160    0.160    0.160    0.160 3018963534.py:15(translate_message_to_morse)
       16    0.000    0.000    0.000    0.000 3018963534.py:5(translate_word_to_morse)
        1    0.000    0.000    0.160    0.160 <string>:1(<module>)
        1    0.000    0.000    0.160    0.160 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.001    0.001    0.001    0.001 {method 'split' of 'str' objects}
       17    0.000    0.000    0.000    0.000 {method 'strip' of 'str' objects}


